# 01 — Bronze Ingest (KaBuM) — ADLS JSONL → Delta (Unity Catalog)

Este notebook lê os arquivos **JSONL** do ADLS (camada *inbox*), cria um DataFrame Bronze e grava em **Delta Lake**, registrando a tabela no **Unity Catalog**.

**Fonte (ADLS / inbox):**  
`abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/`

**Destino (Delta / bronze):**  
`abfss://bronze@storagekabumdata.dfs.core.windows.net/kabum/bronze/products_delta`

**Tabela UC:**  
`workspace.kabum_project.products_bronze`


In [0]:
%run ./00_config_uc


# 00 — Config (Unity Catalog + ADLS)

Centraliza catálogo/schema, storage account e paths base.


In [0]:
# ==========================================
# 0) CATALOG / SCHEMA
# ==========================================

spark.sql("USE CATALOG projeto_data_engineering")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS kabum_project
""")

spark.sql("USE SCHEMA kabum_project")

# ==========================================
# 1) PATHS
# ==========================================
BRONZE_SOURCE_PATH = "abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/"
BRONZE_DELTA_PATH  = "abfss://bronze@storagekabumdata.dfs.core.windows.net/kabum/bronze/products_delta"

print("BRONZE_SOURCE_PATH:", BRONZE_SOURCE_PATH)
print("BRONZE_DELTA_PATH :", BRONZE_DELTA_PATH)


BRONZE_SOURCE_PATH: abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/
BRONZE_DELTA_PATH : abfss://bronze@storagekabumdata.dfs.core.windows.net/kabum/bronze/products_delta


In [0]:
# ==========================================
# 2) VALIDAR ARQUIVOS NA FONTE (JSONL)
# ==========================================
files = dbutils.fs.ls(BRONZE_SOURCE_PATH)
display(files)

# Checagem rápida: quantos arquivos e extensões
exts = {}
for f in files:
    name = f.name
    ext = name.split(".")[-1].lower() if "." in name else ""
    exts[ext] = exts.get(ext, 0) + 1
print("Arquivos:", len(files), "| Extensões:", exts)


path,name,size,modificationTime
abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/data_gamer.jsonl,data_gamer.jsonl,20073,1772654485000
abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/data_i5.jsonl,data_i5.jsonl,93665,1772654486000
abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/data_notebook.jsonl,data_notebook.jsonl,18765,1772654485000
abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/data_ryzen.jsonl,data_ryzen.jsonl,94695,1772654486000
abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/data_ultrabook.jsonl,data_ultrabook.jsonl,17535,1772654485000


Arquivos: 5 | Extensões: {'jsonl': 5}


In [0]:
# ==========================================
# 3) LER JSONL (cada linha = um JSON)
#    OBS: para JSONL, NÃO use multiline=true
# ==========================================
from pyspark.sql import functions as F

df_bronze = (
    spark.read
         .format("json")
         .load(BRONZE_SOURCE_PATH)
         .withColumn("_src_file", F.expr("_metadata.file_path"))
         .withColumn("ingestion_date", F.current_date())
)

display(df_bronze.limit(50))


available,brand,category,currency,discount_percentage,image,kabum_code,marketplace,max_installment,old_price_value,price_value,product_url,rating_value,reviews_count,search_term,title,_src_file,ingestion_date
true,"List(205, https://images4.kabum.com.br/produtos/fabricantes/logo-lenovo.jpg, Lenovo)",Computadores/Notebooks/Notebook Lenovo,BRL,10.0,https://images.kabum.com.br/produtos/fotos/955187/notebook-lenovo-ideapad-slim-3-amd-ryzen-7-7735hs-16gb-ram-amd-radeon-graphics-ssd-512gb-15-3-wuxga-linux-luna-grey-83mms00000_1765997308_original.jpg,955187,kabum,"10x de R$ 388,77",4443.33,3999.0,https://www.kabum.com.br/produto/955187/notebook-lenovo-ideapad-slim-3-amd-ryzen-7-7735hs-16gb-ram-amd-radeon-graphics-ssd-512gb-15-3-wuxga-linux-luna-grey-83mms00000,5.0,76,notebook ryzen,"Notebook Lenovo IdeaPad Slim 3, AMD Ryzen 7 7735HS, 16GB RAM, AMD Radeon Graphics, SSD 512GB, 15.3"" WUXGA, Linux, Luna Grey - 83MMS00000",abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/data_ryzen.jsonl,2026-03-05
true,"List(33, https://images4.kabum.com.br/produtos/fabricantes/logo-acer.jpg, Acer)",Computadores/Notebooks/Notebook Gamer/Notebook Gamer Acer,BRL,10.0,https://images.kabum.com.br/produtos/fotos/sync_mirakl/952820/xlarge/Notebook-Gamer-Acer-Nitro-V15-AMD-Ryzen-7-7735HS-16GB-RAM-RTX-4050-SSD-512-GB-Tela-15-6-Full-HD-Linux-64-Bits-Anv15-41-R2gt_1763054924.jpg,952820,kabum,"10x de R$ 589,90",6400.0,5760.0,https://www.kabum.com.br/produto/952820/notebook-gamer-acer-nitro-v15-anv15-41-r2gt-amd-ryzen-7-7735hs-rtx-4050-16-gb-ram-512-gb-ssd-full-hd-15-6-,5.0,21,notebook ryzen,"Notebook Gamer Acer Nitro V15, AMD Ryzen 7-7735HS, 16GB RAM, RTX 4050, SSD 512 GB, Tela 15.6"" Full HD, Linux 64 Bits - Anv15-41-R2gt",abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/data_ryzen.jsonl,2026-03-05
true,"List(33, https://images4.kabum.com.br/produtos/fabricantes/logo-acer.jpg, Acer)",Computadores/Notebooks/Notebook Acer,BRL,10.0,https://images.kabum.com.br/produtos/fotos/952233/notebook-acer-aspire-5-a515-45-r478-amd-ryzen-5-5500u-16gb-ram-512gb-ssd-15-6-fhd-led-ips-60hz-linux-gutta-nx-aydal-00r_1763390917_original.jpg,952233,kabum,"10x de R$ 344,33",4464.0,4017.6,https://www.kabum.com.br/produto/952233/notebook-acer-aspire-5-a515-45-r478-amd-ryzen-5-5500u-16gb-ram-512gb-ssd-15-6-fhd-led-ips-60hz-linux-gutta-nx-aydal-00r,5.0,10,notebook ryzen,"Notebook Acer Aspire 5 A515-45-R478 AMD Ryzen 5- 5500U, 16GB RAM, 512GB SSD, 15,6"""" FHD Led, IPS, 60Hz, Linux Gutta - NX.AYDAL.00R",abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/data_ryzen.jsonl,2026-03-05
true,"List(1, https://images4.kabum.com.br/produtos/fabricantes/logo-asus.jpg, ASUS)",Computadores/Notebooks/Notebook Asus,BRL,10.0,https://images.kabum.com.br/produtos/fotos/777078/notebook-asus-vivobook-15-amd-ryzen-7-16-gb-512gb-keepos-tela-15-6-fhd-prata-m1502ya-nj612_1746623894_original.jpg,777078,kabum,"10x de R$ 355,44",5248.0,4723.2,https://www.kabum.com.br/produto/777078/notebook-asus-vivobook-15-m1502-amd-ryzen-7-5825u-16gb-ram-512-gb-ssd-keepos-tela-15-6-fhd-cool-silver-m1502ya-nj612,4.0,30,notebook ryzen,"Notebook ASUS Vivobook 15 M1502, AMD Ryzen 7 5825U, 16GB RAM, 512 GB SSD, KeepOS, Tela 15.6, FHD, Cool Silver - M1502YA-NJ612",abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/data_ryzen.jsonl,2026-03-05
true,"List(205, https://images4.kabum.com.br/produtos/fabricantes/logo-lenovo.jpg, Lenovo)",Computadores/Notebooks/Notebook Lenovo,BRL,14.0,https://images.kabum.com.br/produtos/fotos/sync_mirakl/902707/xlarge/Notebook-Lenovo-Ideapad-1-15amn7-Amd-Ryzen-3-7320u-8GB-256gb-SSD-WINDOWS-11-15-6-82x5000mbr-Cloud-Grey_1770929881.jpg,902707,kabum,"10x de R$ 344,88",0.0,2966.01,https://www.kabum.com.br/produto/902707/notebook-lenovo-ideapad-1-15amn7-amd-ryzen-3-7320u-8gb-256gb-ssd-windows-11-15-6-82x5000mbr-cloud-grey,5.0,1,notebook ryzen,"Notebook Lenovo Ideapad 1 15amn7 Amd Ryzen 3 

In [0]:
# ==========================================
# 4) INSPEÇÃO RÁPIDA (schema + contagens)
# ==========================================
df_bronze.printSchema()
print("Rows    :", df_bronze.count())
print("Columns :", len(df_bronze.columns))

# Opcional: checar presença de colunas importantes (ajuste conforme seu schema)
expected_cols = ["marketplace","search_term","product_url","product_name","price","brand"]
present = {c: (c in df_bronze.columns) for c in expected_cols}
print("Expected columns present:", present)


root
 |-- available: boolean (nullable = true)
 |-- brand: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- img: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- image: string (nullable = true)
 |-- kabum_code: long (nullable = true)
 |-- marketplace: string (nullable = true)
 |-- max_installment: string (nullable = true)
 |-- old_price_value: double (nullable = true)
 |-- price_value: double (nullable = true)
 |-- product_url: string (nullable = true)
 |-- rating_value: double (nullable = true)
 |-- reviews_count: long (nullable = true)
 |-- search_term: string (nullable = true)
 |-- title: string (nullable = true)
 |-- _src_file: string (nullable = false)
 |-- ingestion_date: date (nullable = false)

Rows    : 260
Columns : 18
Expected columns present: {'marketplace': True, 'search_term': True, 'product_url': T

In [0]:
# ==========================================
# 5) GRAVAR BRONZE EM DELTA
# ==========================================

(
  df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(BRONZE_DELTA_PATH)
)

print("✅ Delta gravado em:", BRONZE_DELTA_PATH)


✅ Delta gravado em: abfss://bronze@storagekabumdata.dfs.core.windows.net/kabum/bronze/products_delta


In [0]:
%sql
-- ==========================================
-- 6) REGISTRAR TABELA NO UNITY CATALOG
-- ==========================================
CREATE TABLE IF NOT EXISTS projeto_data_engineering.kabum_project.products_bronze
USING DELTA
LOCATION 'abfss://bronze@storagekabumdata.dfs.core.windows.net/kabum/bronze/products_delta'


In [0]:
%sql
-- ==========================================
-- 7) VALIDAR TABELA
-- ==========================================
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT product_url) AS distinct_products
FROM projeto_data_engineering.kabum_project.products_bronze;

rows,distinct_products
260,79
